# Track B, step 5: "looks like a known exploit" with Tiger Data vector search

A security engineer looking at a new vulnerability wants to know: **have attackers gone after something like this before?** This notebook answers that with Granite embeddings stored in **Tiger Data** (our Postgres on Tiger Cloud):

1. Embed all 1,484 vulnerabilities on CISA's KEV list (through December 31, 2025) with the same Granite model as step 2.
2. Store them, and the 72,670 vulnerabilities on the 2026 scoreboard, in Tiger Data as vectors (`pgvector`), with Tiger's **DiskANN** similarity index (`pgvectorscale`).
3. One SQL query then returns, for any 2026 vulnerability, its most similar already-exploited ones. The dashboard can show these as "why this ranks high."
4. Test whether "similarity to past exploits" is itself a useful ranking signal on the 2025 backtest.

**Is this leak-free?** Yes. For 2026, every KEV entry is from 2025 or earlier, so it was known before any 2026 snapshot. For the 2025 test in section 5, each vulnerability is compared only with KEV entries added **by the end of its own publish month**.

**Needs:** step 2's cached embeddings in `data/trackb/`, and `DATABASE_URL` (from the environment or `apps/web/.env.local`). The new tables don't touch the existing `exploit_signals` table.

## 1. Setup

In [1]:
import os, re, time, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_colwidth", 110)
DATA = next(p for p in [Path("../data"), Path("data")] if p.exists()).resolve()
OUT = DATA / "trackb"
MODEL = "ibm-granite/granite-embedding-small-english-r2"

con = duckdb.connect()
for t in ["vulnerabilities", "known_exploited"]:
    con.sql(f'''CREATE VIEW "{t}" AS
                SELECT * FROM read_csv('{DATA}/{t}.csv', quote='"', escape='"', sample_size=-1)''')

def database_url():
    url = os.environ.get("DATABASE_URL")
    env_file = DATA.parent / "apps/web/.env.local"
    if not url and env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.strip().startswith("DATABASE_URL="):
                url = line.split("=", 1)[1].strip().strip('"').strip("'")
    assert url, "Set DATABASE_URL or add it to apps/web/.env.local"
    return url

## 2. Embed the KEV list

We use the same text recipe as steps 1-3 (`assigner | description`, with "exploited in the wild" wording removed), so the KEV vectors and the 2026 vectors are directly comparable. 1,484 short texts take about a minute on a CPU.

In [2]:
IN_WILD = re.compile(r"exploited in the wild|actively exploited|may have been exploited|aware of (?:a )?report", re.IGNORECASE)
def clean(description):
    text = re.sub(r",?\s*as (?:actively )?exploited in the wild[^.]*", "", description or "", flags=re.IGNORECASE)
    return " ".join(s for s in re.split(r"(?<=[.!?])\s+", text) if not IN_WILD.search(s)).strip()

kev = con.sql('''
    SELECT k.cve_id, k.vendor, k.product, CAST(k.date_added AS DATE) AS date_added,
           CAST(v.published AS DATE) AS published, v.assigner,
           COALESCE(v.description, k.short_description) AS description
    FROM known_exploited k LEFT JOIN vulnerabilities v USING (cve_id)
    ORDER BY k.cve_id
''').df()
for c in ["date_added", "published"]:
    kev[c] = pd.to_datetime(kev[c]).dt.date  # plain dates for Postgres
kev["text"] = kev["assigner"].fillna("unknown") + " | " + kev["description"].fillna("").map(clean)
print(f"{len(kev):,} KEV entries | {kev.assigner.isna().sum()} not in vulnerabilities.csv (CISA's description used)")

from sentence_transformers import SentenceTransformer
st = SentenceTransformer(MODEL, device="cpu")
st.max_seq_length = 256
t0 = time.time()
kev_emb = st.encode(kev["text"].tolist(), batch_size=64, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
print(f"embedded in {time.time() - t0:.0f} s, shape {kev_emb.shape}")

s26_emb = np.load(OUT / "emb_score26.npy")
s26 = pd.read_parquet(OUT / "emb_score26_ids.parquet")
s26 = s26.merge(con.sql("SELECT cve_id, CAST(published AS DATE) AS published, description FROM vulnerabilities").df(),
                on="cve_id", how="left")
s26["published"] = pd.to_datetime(s26["published"]).dt.date
assert len(s26) == len(s26_emb) == 72_670

1,484 KEV entries | 0 not in vulnerabilities.csv (CISA's description used)


embedded in 80 s, shape (1484, 384)


## 3. Load the vectors into Tiger Data

- **`pgvector`** adds a `vector(384)` column type and the `<=>` operator, cosine distance: 0 means identical meaning and 2 means opposite. *Similarity* is 1 minus that.
- **`pgvectorscale`** is Tiger's extension. It adds the **DiskANN** index, which finds nearest neighbors without comparing against every row. It's the same idea as a book index, for "meaning" instead of words. We index the KEV table, the one the dashboard searches. The 2026 table is only looked up by `cve_id`, which its primary key already handles.

Loading uses binary `COPY`, the fastest way to bulk-load Postgres. The tables are emptied first, so re-running this notebook gives the same result.

In [3]:
import psycopg
from pgvector.psycopg import register_vector

t0 = time.time()
with psycopg.connect(database_url(), autocommit=True) as db:
    db.execute("CREATE EXTENSION IF NOT EXISTS vector")
    db.execute("CREATE EXTENSION IF NOT EXISTS vectorscale CASCADE")
    register_vector(db)
    db.execute('''CREATE TABLE IF NOT EXISTS kev_embeddings (
        cve_id text PRIMARY KEY, vendor text, product text, date_added date, published date,
        description text, embedding vector(384) NOT NULL)''')
    db.execute('''CREATE TABLE IF NOT EXISTS scoreboard_embeddings (
        cve_id text PRIMARY KEY, published date, description text, embedding vector(384) NOT NULL)''')
    db.execute("TRUNCATE kev_embeddings, scoreboard_embeddings")
    with db.cursor() as cur:
        with cur.copy("COPY kev_embeddings FROM STDIN WITH (FORMAT BINARY)") as cp:
            cp.set_types(["text", "text", "text", "date", "date", "text", "vector"])
            for r, e in zip(kev.itertuples(), kev_emb):
                cp.write_row([r.cve_id, r.vendor, r.product, r.date_added,
                              None if pd.isna(r.published) else r.published, r.description, e])
        with cur.copy("COPY scoreboard_embeddings FROM STDIN WITH (FORMAT BINARY)") as cp:
            cp.set_types(["text", "date", "text", "vector"])
            for r, e in zip(s26.itertuples(), s26_emb):
                cp.write_row([r.cve_id, None if pd.isna(r.published) else r.published, r.description, e])
    # Only the KEV table is searched by similarity; 2026 rows are looked up by cve_id (primary key).
    # (A DiskANN index on all 72,670 2026 rows also exceeded this small service's memory.)
    db.execute("CREATE INDEX IF NOT EXISTS kev_embeddings_diskann ON kev_embeddings USING diskann (embedding vector_cosine_ops)")
    db.execute("DROP INDEX IF EXISTS scoreboard_embeddings_diskann")
    db.execute("ANALYZE kev_embeddings"); db.execute("ANALYZE scoreboard_embeddings")
    counts = {t: db.execute(f"SELECT count(*) FROM {t}").fetchone()[0] for t in ["kev_embeddings", "scoreboard_embeddings"]}
    ext = db.execute("SELECT extname, extversion FROM pg_extension WHERE extname IN ('vector','vectorscale')").fetchall()
print(f"loaded in {(time.time() - t0) / 60:.1f} min:", counts, "| extensions:", dict(ext))
assert counts == {"kev_embeddings": len(kev), "scoreboard_embeddings": len(s26)}

loaded in 0.6 min: {'kev_embeddings': 1484, 'scoreboard_embeddings': 72670} | extensions: {'vector': '0.8.7', 'vectorscale': '0.9.0'}


## 4. The dashboard query: "most similar known exploits"

Given a 2026 vulnerability, look up its vector and return the 5 closest KEV entries. This is the exact SQL the dashboard can run.

In [4]:
SIMILAR_SQL = '''
    SELECT k.cve_id, k.vendor, k.product, k.date_added,
           round((1 - (k.embedding <=> q.embedding))::numeric, 3) AS similarity,
           left(k.description, 90) AS description
    FROM scoreboard_embeddings q, kev_embeddings k
    WHERE q.cve_id = %s
    ORDER BY k.embedding <=> q.embedding
    LIMIT 5
'''

def similar_known_exploits(db, cve_id):
    return pd.DataFrame(db.execute(SIMILAR_SQL, [cve_id]).fetchall(),
                        columns=["kev_cve_id", "vendor", "product", "added_to_kev", "similarity", "description"])

# Examples: one 2026 vulnerability each for a few product types that attackers often target.
examples = []
for pattern in [r"FortiOS|FortiGate", r"Ivanti", r"Chrome.*V8|V8.*Chrome", r"Citrix|NetScaler"]:
    hit = s26[s26.description.str.contains(pattern, case=False, na=False, regex=True)]
    if len(hit):
        examples.append(hit.iloc[0].cve_id)

with psycopg.connect(database_url()) as db:
    for cve in examples:
        desc = s26.set_index("cve_id").at[cve, "description"]
        t0 = time.time()
        res = similar_known_exploits(db, cve)
        print(f"\n{cve}: {desc[:150]}...\n  ({(time.time() - t0) * 1000:.0f} ms)")
        print(res.to_string(index=False))


CVE-2025-25249: A heap-based buffer overflow vulnerability in Fortinet FortiOS 7.6.0 through 7.6.3, FortiOS 7.4.0 through 7.4.8, FortiOS 7.2.0 through 7.2.11, FortiOS...
  (495 ms)
    kev_cve_id   vendor                        product added_to_kev similarity                                                                                description
CVE-2023-27997 Fortinet FortiOS and FortiProxy SSL-VPN   2023-06-13      0.972 A heap-based buffer overflow vulnerability [CWE-122] in FortiOS version 7.2.4 and below, v
CVE-2022-42475 Fortinet                        FortiOS   2022-12-13      0.962 A heap-based buffer overflow vulnerability [CWE-122] in FortiOS SSL-VPN 7.2.0 through 7.2.
CVE-2024-21762 Fortinet                        FortiOS   2024-02-09      0.958 A out-of-bounds write vulnerability in Fortinet FortiOS 7.4.0 through 7.4.2, FortiOS 7.2.0
CVE-2024-23113 Fortinet              Multiple Products   2024-10-09      0.939 A use of externally-controlled format string in Fortinet Fo


CVE-2026-10523: An Authentication Bypass vulnerability (CWE-288) in Ivanti Sentry before the R10.5.2, R10.6.2 and R10.7.1 versions allows a remote unauthenticated att...
  (220 ms)
    kev_cve_id vendor                          product added_to_kev similarity                                                                                description
 CVE-2024-8963 Ivanti   Cloud Services Appliance (CSA)   2024-09-19      0.925 Path Traversal in the Ivanti CSA before 4.6 Patch 519 allows a remote unauthenticated atta
 CVE-2024-9380 Ivanti   Cloud Services Appliance (CSA)   2024-10-09      0.923 An OS command injection vulnerability in the admin web console of Ivanti CSA before versio
 CVE-2024-7593 Ivanti          Virtual Traffic Manager   2024-09-24      0.917 Incorrect implementation of an authentication algorithm in Ivanti vTM other than versions 
CVE-2023-46805 Ivanti Connect Secure and Policy Secure   2024-01-10      0.910 An authentication bypass vulnerability in the web component


CVE-2026-0899: Out of bounds memory access in V8 in Google Chrome prior to 144.0.7559.59 allowed a remote attacker to potentially exploit object corruption via a cra...
  (279 ms)
    kev_cve_id vendor     product added_to_kev similarity                                                                                description
 CVE-2024-0519 Google Chromium V8   2024-01-17      0.994 Out of bounds memory access in V8 in Google Chrome prior to 120.0.6099.224 allowed a remot
 CVE-2025-5419 Google Chromium V8   2025-06-05      0.980 Out of bounds read and write in V8 in Google Chrome prior to 137.0.7151.68 allowed a remot
CVE-2025-14174 Google    Chromium   2025-12-12      0.976 Out of bounds memory access in ANGLE in Google Chrome on Mac prior to 143.0.7499.110 allow
 CVE-2024-4761 Google Chromium V8   2024-05-16      0.973 Out of bounds write in V8 in Google Chrome prior to 124.0.6367.207 allowed a remote attack
CVE-2021-30632 Google Chromium V8   2021-11-03      0.964 Out of bounds wr


CVE-2025-66391: In Citrix Cloud through 2025-11-10, an account with read-only access can trigger the beginning of a workflow for write operations, e.g., the system wi...
  (221 ms)
    kev_cve_id vendor                                           product added_to_kev similarity                                                                                description
 CVE-2024-8069 Citrix                                 Session Recording   2025-08-25      0.877 Limited remote code execution with privilege of a NetworkService Account access in Citrix 
 CVE-2024-8068 Citrix                                 Session Recording   2025-08-25      0.869 Privilege escalation to NetworkService Account access in Citrix Session Recording when an 
CVE-2019-11634 Citrix    Workspace Application and Receiver for Windows   2021-11-03      0.859                 Citrix Workspace App before 1904 for Windows has Incorrect Access Control.
 CVE-2023-3519 Citrix               NetScaler ADC and NetScaler Gatewa

**Reading this: the neighbors make sense to a security engineer.**
- A 2026 FortiOS heap overflow lands next to the famous FortiOS SSL-VPN heap overflows CVE-2023-27997 and CVE-2022-42475 (similarity 0.97 and 0.96).
- An Ivanti Sentry authentication bypass lands next to earlier exploited Ivanti appliance bugs.
- A Chrome V8 out-of-bounds bug lands next to past V8 zero-days (up to 0.99).

Each lookup takes about 200 ms from a laptop, mostly network time. That's fast enough to show live in the dashboard.

**Check that the database agrees with a plain NumPy search.** With normalized vectors, cosine similarity is just a dot product. If Tiger's neighbors match NumPy's, the load and the query are correct.

In [5]:
idx = s26.reset_index().set_index("cve_id")["index"]
with psycopg.connect(database_url()) as db:
    for cve in examples:
        sims = kev_emb @ s26_emb[idx[cve]]
        numpy_top = kev["cve_id"].to_numpy()[np.argsort(-sims)[:5]].tolist()
        tiger_top = similar_known_exploits(db, cve)["kev_cve_id"].tolist()
        print(cve, "match" if numpy_top == tiger_top else f"DIFFERENT: {numpy_top} vs {tiger_top}")

CVE-2025-25249 match


CVE-2026-10523 match


CVE-2026-0899 match


CVE-2025-66391 match


All four examples return the same five neighbors from Tiger Data as from the exact NumPy search, so the stored vectors and the query are correct.

## 5. Is "looks like a known exploit" a ranking signal?

For each 2025 validation vulnerability, compute its **highest similarity to any KEV entry added by the end of its publish month** (point-in-time, so no future KEV entries). Then rank by that and score it on the same backtest as everything else.

*Example:* a February 2025 FortiOS bug is compared only with KEV entries added by February 28, 2025.

In [6]:
val = pd.read_parquet(DATA / "backtest/val_2025.parquet").sort_values("cve_id", ignore_index=True)
val_ids = pd.read_parquet(OUT / "emb_val_ids.parquet")["cve_id"]
assert val_ids.equals(val["cve_id"])
val_emb = np.load(OUT / "emb_val.npy")

sims = val_emb @ kev_emb.T                                  # 35,281 x 1,484 cosine similarities
snap_end = (pd.to_datetime(val["snap"]) + pd.offsets.MonthEnd(0)).to_numpy()
allowed = pd.to_datetime(kev["date_added"]).to_numpy()[None, :] <= snap_end[:, None]
sims[~allowed] = -1                                         # hide KEV entries added after the snapshot
val["kev_similarity"] = sims.max(axis=1)
val["nearest_kev"] = kev["cve_id"].to_numpy()[sims.argmax(axis=1)]
print(f"median KEV entries visible per row: {int(np.median(allowed.sum(axis=1)))}")

median KEV entries visible per row: 1351


In [7]:
BENCH_K, BENCH_N = 500, 72_670
default_k = lambda n: int(round(n * BENCH_K / BENCH_N))
def rank_order(df, score, seed=0):
    cols = [score] if isinstance(score, str) else score
    keys = [np.where(np.isnan(df[c].to_numpy(float)), -np.inf, df[c].to_numpy(float)) for c in cols]
    tie = np.random.default_rng(seed).permutation(len(df))
    return np.lexsort([tie] + [-k for k in reversed(keys)])
def hits_at_k(df, score, k=None):
    k = k or default_k(len(df))
    return int(df["y"].to_numpy()[rank_order(df, score)[:k]].sum())

frozen = pd.read_csv(OUT / "text_val_2025_frozen.csv").set_index("cve_id")["text_score"]
val["frozen_text"] = val["cve_id"].map(frozen)
rows = []
for name, s in [("Exploit code then EPSS (baseline)", ["any_exploit", "epss"]), ("EPSS first", "epss"),
                ("Frozen Granite text (step 2)", "frozen_text"), ("Similarity to past KEV entries", "kev_similarity")]:
    col = s if isinstance(s, str) else None
    rows.append({"ranking": name, **{f"top {k}": hits_at_k(val, s, k) for k in [243, 500, 1000, 2000]},
                 "AUC": round(float(roc_auc_score(val.y, val[col].fillna(0))), 3) if col else ""})
pd.DataFrame(rows)

,ranking,top 243,top 500,top 1000,top 2000,AUC
0,Exploit code then EPSS (baseline),14,18,22,24,
1,EPSS first,8,8,13,18,0.707
2,Frozen Granite text (step 2),4,6,9,19,0.8
3,Similarity to past KEV entries,1,3,5,8,0.671


**Reading this: as a ranking on its own, similarity is weak.** It finds 1 of 52 in the top 243 and has an AUC of 0.67. That's lower than the frozen text score (0.80), because similarity can't separate bugs *within* a product family. Every Chrome V8 bug looks like a past Chrome zero-day, but only a few of them get exploited.

**Where exploited vulnerabilities sit by similarity.** Exploited vulnerabilities should look more like past exploits than the typical one does.

In [8]:
val["similarity band"] = pd.cut(val["kev_similarity"], [-1, 0.6, 0.7, 0.8, 0.9, 1.0])
val.groupby("similarity band", observed=True).agg(rows=("y", "size"), positives=("y", "sum"),
                                                  rate=("y", lambda s: f"{s.mean():.2%}"))

,rows,positives,rate
similarity band,,,
"(0.8, 0.9]",30575,31,0.10%
"(0.9, 1.0]",4700,21,0.45%


**Reading this:** every 2025 vulnerability is at least 0.8 similar to some KEV entry, since Granite embeddings of security text all sit fairly close together. The informative split is at 0.9. The 4,700 most-similar vulnerabilities (13% of the set) hold 21 of the 52 positives (40%), a 0.45% rate against 0.10% for the rest, or 4.5x. So similarity is a useful **context signal**: "this looks like a known exploit" is worth showing an engineer, even though it can't rank on its own.

In [9]:
top = val[val.y == 1].sort_values("kev_similarity", ascending=False).head(8)
top.merge(kev[["cve_id", "vendor", "product"]].rename(columns={"cve_id": "nearest_kev"}), on="nearest_kev")[
    ["cve_id", "kev_similarity", "nearest_kev", "vendor", "product", "description"]].round(3)

,cve_id,kev_similarity,nearest_kev,vendor,product,description
0,CVE-2025-6554,0.981,CVE-2023-4762,Google,Chromium V8,Type confusion in V8 in Google Chrome prior to 138.0.7204.96 allowed a remote attacker to perform arbitrar...
1,CVE-2025-54236,0.962,CVE-2022-24086,Adobe,Commerce and Magento Open Source,"Adobe Commerce versions 2.4.9-alpha2, 2.4.8-p2, 2.4.7-p7, 2.4.6-p12, 2.4.5-p14, 2.4.4-p15 and earlier are ..."
2,CVE-2025-24472,0.955,CVE-2024-55591,Fortinet,FortiOS and FortiProxy,An Authentication Bypass Using an Alternate Path or Channel vulnerability [CWE-288] affecting FortiOS 7.0....
3,CVE-2025-5777,0.953,CVE-2025-6543,Citrix,NetScaler ADC and Gateway,Insufficient input validation leading to memory overread when the NetScaler is configured as a Gateway (VP...
4,CVE-2025-41244,0.938,CVE-2022-22960,VMware,Multiple Products,VMware Aria Operations and VMware Tools contain a local privilege escalation vulnerability. A malicious lo...
5,CVE-2025-27915,0.932,CVE-2022-27926,Synacor,Zimbra Collaboration Suite (ZCS),An issue was discovered in Zimbra Collaboration (ZCS) 9.0 and 10.0 and 10.1. A stored cross-site scripting...
6,CVE-2025-6218,0.931,CVE-2023-38831,RARLAB,WinRAR,RARLAB WinRAR Directory Traversal Remote Code Execution Vulnerability. This vulnerability allows remote at...
7,CVE-2025-20281,0.929,CVE-2023-20273,Cisco,Cisco IOS XE Web UI,"A vulnerability in a specific API of Cisco ISE and Cisco ISE-PIC could allow an unauthenticated, remote at..."


**Reading this: real "why fix first" stories.** These later-exploited 2025 vulnerabilities each looked like a known exploit at publication:
- CVE-2025-6554 (Chrome V8 type confusion) is 0.98 similar to CVE-2023-4762, a V8 type confusion already on KEV.
- CVE-2025-5777 ("CitrixBleed 2") is closest to CVE-2025-6543, an exploited NetScaler bug.
- CVE-2025-6218 (WinRAR directory traversal) is closest to CVE-2023-38831, the WinRAR bug exploited in 2023.

This is the explanation the dashboard can show next to each ranked vulnerability.

## 6. Save the similarity scores

`kev_similarity_val_2025.csv` holds the point-in-time score for the validation set. `kev_similarity_2026.csv` holds the same for the 2026 scoreboard, against all KEV entries through 2025. Both can be blended or used as a feature like the text score.

In [10]:
val[["cve_id", "kev_similarity"]].to_csv(OUT / "kev_similarity_val_2025.csv", index=False)
s26_sim = (s26_emb @ kev_emb.T).max(axis=1)
pd.DataFrame({"cve_id": s26["cve_id"], "kev_similarity": s26_sim}).to_csv(OUT / "kev_similarity_2026.csv", index=False)
print("Saved kev_similarity_val_2025.csv and kev_similarity_2026.csv")

Saved kev_similarity_val_2025.csv and kev_similarity_2026.csv


## Summary

- **Tiger Data now holds** `kev_embeddings` (1,484 KEV entries, with a DiskANN index from `pgvectorscale`) and `scoreboard_embeddings` (72,670 vulnerabilities from 2026), next to the existing `exploit_signals` table.
- **One SQL query** (`SIMILAR_SQL` above) returns a 2026 vulnerability's five most similar known exploits in about 200 ms. Its results match an exact search.
- **As a ranking, similarity is weak** (1 hit@243, AUC 0.67). **As context, it's strong:** the most-similar 13% hold 40% of the exploited vulnerabilities (4.5x the base rate), and the neighbors are recognizable stories (CitrixBleed 2, WinRAR, Chrome V8).
- **Use in the demo:** show the nearest known exploits next to each top-500 vulnerability as part of "why fix first", and pass them to the watsonx Granite explanation.

*Note:* a DiskANN index on the 72,670-row 2026 table exceeded this Tiger service's memory, and the long build dropped the connection. That table is only looked up by ID, so it doesn't need one.